# Chapter 15: Model Evaluation, Selection & Tuning Assessment


## Assignment Overview

**Business Context**: You're a data analyst working for a hotel chain. Management wants you to build a reliable model to predict whether guests will cancel their reservations. Rather than just fitting a single model, they need you to follow a **professional evaluation workflow**: evaluate models reliably using cross-validation, diagnose model behavior with learning and validation curves, tune hyperparameters systematically, and compare algorithms fairly before selecting a final model.

**Learning Objectives**:
- Understand why a single train/test split is insufficient for reliable model evaluation
- Use cross-validation (StratifiedKFold) to evaluate models with multiple metrics
- Diagnose bias and variance using learning curves
- Identify optimal hyperparameter ranges using validation curves
- Tune hyperparameters systematically using GridSearchCV and RandomizedSearchCV
- Compare multiple algorithms fairly using the same evaluation framework
- Evaluate a final model on a held-out test set
- Interpret performance differences using mean and standard deviation

**Important Notes**:
- This assignment focuses on **Chapter 15 content only** (model evaluation, selection, and tuning)
- You **WILL** split data into train/test sets and freeze the test set for final evaluation only
- All model development uses **cross-validation on the training set**
- Use `random_state=27` for all models and splits (for reproducibility)
- The test set is used **only once** at the very end

---

## Dataset Information

**Source**: Hotel Reservations Classification Dataset (Kaggle)
**Description**:
This dataset contains information about hotel reservations, including booking details, guest information, and reservation characteristics. The goal is to predict whether a reservation will be canceled.

**Target Variable**: `booking_status` - categorical variable indicating whether the reservation was "Canceled" or "Not_Canceled"

**Data Dictionary**:
| Variable | Description |
|---|---|
| Booking_ID | Unique identifier for each reservation |
| no_of_adults | Number of adults in the reservation |
| no_of_children | Number of children in the reservation |
| no_of_weekend_nights | Number of weekend nights (Saturday or Sunday) |
| no_of_week_nights | Number of week nights (Monday to Friday) |
| type_of_meal_plan | Type of meal plan selected (Meal Plan 1, Meal Plan 2, Meal Plan 3, Not Selected) |
| required_car_parking_space | Whether a car parking space is required (0 or 1) |
| room_type_reserved | Type of room reserved (Room_Type 1 through Room_Type 7) |
| lead_time | Number of days between booking date and arrival date |
| arrival_year | Year of arrival (2017 or 2018) |
| arrival_month | Month of arrival (1-12) |
| arrival_date | Day of the month of arrival (1-31) |
| market_segment_type | How the reservation was made (Online, Offline, Corporate, Complementary, Aviation) |
| repeated_guest | Whether the guest has previously stayed at the hotel (0 or 1) |
| no_of_previous_cancellations | Number of previous cancellations by the guest |
| no_of_previous_bookings_not_canceled | Number of previous bookings not canceled by the guest |
| avg_price_per_room | Average price per day of the reservation (in euros) |
| no_of_special_requests | Number of special requests made by the guest |
| booking_status | Target: "Canceled" or "Not_Canceled" |

**Note**: Expected checkpoint answers are based on using `random_state=27` throughout. Actual values may vary slightly if different random seeds are used.

---

# Part 1: Load and Prepare Data

In this section, you'll load the dataset and perform data preprocessing for model evaluation and selection.

## Question 1: Load and Explore the Dataset

Load the `Hotel Reservations.csv` file and perform initial exploration:
1. Load the dataset into a Pandas DataFrame
2. Display the dataset shape and first few rows
3. Check the distribution of the target variable `booking_status`
4. Calculate and display the cancellation rate percentage

**Checkpoint Question**: What percentage of reservations are canceled? Round to 2 decimal places (e.g., enter 32.76 not 0.3276 or 32.76%).


In [ ]:
# Question 1: Load and explore the dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('hotel_reservations.csv')

# Display shape and first few rows
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()


In [ ]:
# Distribution of target variable
print("Target variable distribution:")
print(df['booking_status'].value_counts())
print()

# Cancellation rate
cancellation_rate = (df['booking_status'] == 'Canceled').mean() * 100
print(f"Cancellation rate: {cancellation_rate:.2f}%")


# Question 2: Data preprocessing and train/test split
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Create binary target
df['is_canceled'] = (df['booking_status'] == 'Canceled').astype(int)

# Drop ID and original target
df = df.drop(columns=['Booking_ID', 'booking_status'])

# Identify column types
numeric_cols = df.select_dtypes(include='number').columns.drop('is_canceled').tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()

print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

# Features and target
X = df.drop(columns=['is_canceled'])
y = df['is_canceled']

# Preprocessing pipeline
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# Cross-validation object
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=27)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=27
)

print(f"\nTraining set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"\nTraining samples: {X_train.shape[0]}")


In [ ]:
# Question 2: Data preprocessing and train/test split

# YOUR CODE HERE


# Part 2: Cross-Validation Baselines

In this section, you'll establish baseline performance for two models using cross-validation with multiple metrics.

# Question 3: Logistic regression baseline with multiple metrics
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression

# Scoring dictionary
scoring = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'neg_log_loss': 'neg_log_loss'
}

# Logistic regression pipeline
lr_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('lr', LogisticRegression(max_iter=2000, random_state=27))
])

# Cross-validate
lr_cv = cross_validate(lr_pipe, X_train, y_train, cv=skf, scoring=scoring)

# Display results
print("Logistic Regression Cross-Validation Results:")
print(f"  Accuracy:          {lr_cv['test_accuracy'].mean():.4f} (+/- {lr_cv['test_accuracy'].std():.4f})")
print(f"  Balanced Accuracy: {lr_cv['test_balanced_accuracy'].mean():.4f} (+/- {lr_cv['test_balanced_accuracy'].std():.4f})")
print(f"  F1:                {lr_cv['test_f1'].mean():.4f} (+/- {lr_cv['test_f1'].std():.4f})")
print(f"  ROC AUC:           {lr_cv['test_roc_auc'].mean():.4f} (+/- {lr_cv['test_roc_auc'].std():.4f})")
print(f"  Log Loss:          {-lr_cv['test_neg_log_loss'].mean():.4f} (+/- {lr_cv['test_neg_log_loss'].std():.4f})")


In [ ]:
# Question 3: Logistic regression baseline with multiple metrics

# YOUR CODE HERE


# Question 4: Decision tree baseline with cross-validation
from sklearn.tree import DecisionTreeClassifier

# Decision tree pipeline
dt_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('tree', DecisionTreeClassifier(max_depth=5, random_state=27))
])

# Cross-validate
dt_cv = cross_validate(dt_pipe, X_train, y_train, cv=skf, scoring=scoring)

# Display results
print("Decision Tree Cross-Validation Results:")
print(f"  Accuracy:          {dt_cv['test_accuracy'].mean():.4f} (+/- {dt_cv['test_accuracy'].std():.4f})")
print(f"  Balanced Accuracy: {dt_cv['test_balanced_accuracy'].mean():.4f} (+/- {dt_cv['test_balanced_accuracy'].std():.4f})")
print(f"  F1:                {dt_cv['test_f1'].mean():.4f} (+/- {dt_cv['test_f1'].std():.4f})")
print(f"  ROC AUC:           {dt_cv['test_roc_auc'].mean():.4f} (+/- {dt_cv['test_roc_auc'].std():.4f})")
print(f"  Log Loss:          {-dt_cv['test_neg_log_loss'].mean():.4f} (+/- {dt_cv['test_neg_log_loss'].std():.4f})")


In [ ]:
# Question 4: Decision tree baseline with cross-validation

# YOUR CODE HERE


# Part 3: Diagnostic Curves

In this section, you'll use learning curves to diagnose whether the logistic regression model suffers from high bias or high variance.

# Question 5: Learning curve for logistic regression
from sklearn.model_selection import learning_curve

# Compute learning curve
train_sizes, train_scores, val_scores = learning_curve(
    lr_pipe, X_train, y_train, cv=skf, scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', label='Training ROC AUC', color='blue')
plt.plot(train_sizes, val_mean, 'o-', label='Validation ROC AUC', color='orange')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='orange')
plt.xlabel('Training Set Size')
plt.ylabel('ROC AUC')
plt.title('Learning Curve - Logistic Regression')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

# Print final values
print(f"Final Training ROC AUC:   {train_mean[-1]:.4f}")
print(f"Final Validation ROC AUC: {val_mean[-1]:.4f}")
print(f"Gap: {train_mean[-1] - val_mean[-1]:.4f}")
print("\nInterpretation: The model shows HIGH BIAS. Both curves converge quickly at ~0.86")
print("with a very small gap (~0.001), indicating the model is underfitting.")
print("The linear model cannot capture more complex patterns in the data.")


In [ ]:
# Question 5: Learning curve for logistic regression

# YOUR CODE HERE


# Part 4: Validation Curves

In this section, you'll use validation curves to identify the optimal hyperparameter value for a decision tree.

# Question 6: Validation curve for decision tree (max_depth)
from sklearn.model_selection import validation_curve

# Decision tree pipeline (without fixed max_depth)
dt_pipe_vc = Pipeline([
    ('preprocessor', preprocessor),
    ('tree', DecisionTreeClassifier(random_state=27))
])

# Compute validation curve
param_range = np.arange(1, 21)
train_scores_vc, val_scores_vc = validation_curve(
    dt_pipe_vc, X_train, y_train, param_name='tree__max_depth',
    param_range=param_range, cv=skf, scoring='roc_auc', n_jobs=-1
)

train_mean_vc = train_scores_vc.mean(axis=1)
val_mean_vc = val_scores_vc.mean(axis=1)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(param_range, train_mean_vc, 'o-', label='Training ROC AUC', color='blue')
plt.plot(param_range, val_mean_vc, 'o-', label='Validation ROC AUC', color='orange')
plt.xlabel('max_depth')
plt.ylabel('ROC AUC')
plt.title('Validation Curve - Decision Tree (max_depth)')
plt.legend(loc='best')
plt.grid(True)
plt.xticks(param_range)
plt.tight_layout()
plt.show()

# Find optimal depth
best_depth = param_range[np.argmax(val_mean_vc)]
print(f"Optimal max_depth: {best_depth}")
print(f"Best validation ROC AUC: {val_mean_vc.max():.4f}")


In [ ]:
# Question 6: Validation curve for decision tree (max_depth)

# YOUR CODE HERE


# Part 5: Hyperparameter Tuning

In this section, you'll use GridSearchCV and RandomizedSearchCV to systematically find optimal hyperparameters.

# Question 7: GridSearchCV for logistic regression
from sklearn.model_selection import GridSearchCV

# Logistic regression pipeline
lr_pipe_grid = Pipeline([
    ('preprocessor', preprocessor),
    ('lr', LogisticRegression(max_iter=2000, random_state=27))
])

# Parameter grid
param_grid = {
    'lr__C': [0.01, 0.1, 1.0, 10.0],
    'lr__l1_ratio': [0.0, 1.0],
    'lr__penalty': ['elasticnet'],
    'lr__solver': ['saga']
}

# GridSearchCV
grid_search = GridSearchCV(lr_pipe_grid, param_grid, cv=skf, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("GridSearchCV Results:")
print(f"  Best parameters: {grid_search.best_params_}")
print(f"  Best ROC AUC: {grid_search.best_score_:.4f}")


In [ ]:
# Question 7: GridSearchCV for logistic regression

# YOUR CODE HERE


# Question 8: RandomizedSearchCV for gradient boosting
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier

# Gradient boosting pipeline
gb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('gb', GradientBoostingClassifier(random_state=27))
])

# Parameter distributions
param_dist = {
    'gb__n_estimators': [50, 100, 200],
    'gb__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'gb__max_depth': [2, 3, 4, 5]
}

# RandomizedSearchCV
random_search = RandomizedSearchCV(
    gb_pipe, param_dist, n_iter=20, cv=skf,
    scoring='roc_auc', random_state=27, n_jobs=-1
)
random_search.fit(X_train, y_train)

print("RandomizedSearchCV Results:")
print(f"  Best parameters: {random_search.best_params_}")
print(f"  Best ROC AUC: {random_search.best_score_:.4f}")


In [ ]:
# Question 8: RandomizedSearchCV for gradient boosting

# YOUR CODE HERE


# Part 6: Fair Model Comparison

In this section, you'll cross-validate multiple models under identical conditions and build a comparison table.

# Question 9: Cross-validate multiple models
from sklearn.ensemble import RandomForestClassifier

# Define models
models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('clf', LogisticRegression(max_iter=2000, random_state=27))
    ]),
    'Decision Tree': Pipeline([
        ('preprocessor', preprocessor),
        ('clf', DecisionTreeClassifier(max_depth=5, random_state=27))
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=27, n_jobs=-1))
    ]),
    'Gradient Boosting': Pipeline([
        ('preprocessor', preprocessor),
        ('clf', GradientBoostingClassifier(n_estimators=100, random_state=27))
    ])
}

# Cross-validate all models
results_list = []
print("Model Comparison (Cross-Validated ROC AUC):")
print("-" * 55)
for name, pipe in models.items():
    cv_results = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring)
    roc_mean = cv_results['test_roc_auc'].mean()
    roc_std = cv_results['test_roc_auc'].std()
    print(f"  {name:25s}: {roc_mean:.4f} (+/- {roc_std:.4f})")
    results_list.append({
        'Model': name,
        'Accuracy_mean': cv_results['test_accuracy'].mean(),
        'Accuracy_std': cv_results['test_accuracy'].std(),
        'F1_mean': cv_results['test_f1'].mean(),
        'F1_std': cv_results['test_f1'].std(),
        'ROC_AUC_mean': cv_results['test_roc_auc'].mean(),
        'ROC_AUC_std': cv_results['test_roc_auc'].std(),
        'Log_Loss_mean': -cv_results['test_neg_log_loss'].mean(),
        'Log_Loss_std': cv_results['test_neg_log_loss'].std()
    })

print(f"\nHighest ROC AUC: Random Forest")


In [ ]:
# Question 9: Cross-validate multiple models

# YOUR CODE HERE


# Question 10: Build comparison table
results_df = pd.DataFrame(results_list)

# Create formatted columns
results_df['Accuracy'] = results_df.apply(
    lambda r: f"{r['Accuracy_mean']:.4f} +/- {r['Accuracy_std']:.4f}", axis=1)
results_df['F1'] = results_df['F1_mean'].apply(lambda x: f"{x:.4f}")
results_df['ROC AUC'] = results_df.apply(
    lambda r: f"{r['ROC_AUC_mean']:.4f} +/- {r['ROC_AUC_std']:.4f}", axis=1)
results_df['Log Loss'] = results_df['Log_Loss_mean'].apply(lambda x: f"{x:.4f}")

# Sort by ROC AUC descending
results_df = results_df.sort_values('ROC_AUC_mean', ascending=False)

# Display formatted table
display_df = results_df[['Model', 'Accuracy', 'F1', 'ROC AUC', 'Log Loss']].reset_index(drop=True)
display(display_df)

# Print RF F1 for checkpoint
rf_f1 = results_df[results_df['Model'] == 'Random Forest']['F1_mean'].values[0]
print(f"\nRandom Forest F1 score: {rf_f1:.4f}")


In [ ]:
# Question 10: Build comparison table

# YOUR CODE HERE


# Part 7: Final Evaluation

In this section, you'll evaluate the best models on the held-out test set for the first and only time.

# Question 11: Evaluate best model on held-out test set
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, f1_score

# Tuned Gradient Boosting (best params from Q8)
gb_tuned = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.2, random_state=27
    ))
])

# Default Random Forest
rf_default = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=27, n_jobs=-1))
])

# Train on full training set
gb_tuned.fit(X_train, y_train)
rf_default.fit(X_train, y_train)

# Evaluate on test set
print("Test Set Evaluation:")
print("=" * 65)
for name, model in [('Tuned Gradient Boosting', gb_tuned), ('Default Random Forest', rf_default)]:
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    ll = log_loss(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)
    print(f"\n{name}:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  ROC AUC:  {roc:.4f}")
    print(f"  Log Loss: {ll:.4f}")
    print(f"  F1:       {f1:.4f}")


In [ ]:
# Question 11: Evaluate best model on held-out test set

# YOUR CODE HERE


# Question 12: Confusion matrix and classification metrics
from sklearn.metrics import confusion_matrix, precision_score, recall_score
import seaborn as sns

# Predictions from tuned GB
y_pred_gb = gb_tuned.predict(X_test)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_gb)

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Canceled', 'Canceled'],
            yticklabels=['Not Canceled', 'Canceled'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Tuned Gradient Boosting')
plt.tight_layout()
plt.show()

# Precision and recall for canceled class
precision = precision_score(y_test, y_pred_gb)
recall = recall_score(y_test, y_pred_gb)

print(f"Precision (canceled class): {precision:.4f}")
print(f"Recall (canceled class):    {recall:.4f}")


In [ ]:
# Question 12: Confusion matrix and classification metrics

# YOUR CODE HERE


# Part 8: Conceptual Understanding (Multiple Choice Questions)

Answer the following multiple choice questions to demonstrate your understanding of model evaluation, selection, and tuning from Chapter 15. Each question references your analysis results and tests Chapter 15 concepts.

Select the **best answer** for each question (A, B, C, D, or E).

---

In [ ]:
# Question 13: Cross-Validation Purpose
# Answer: B
# Cross-validation produces multiple performance estimates, revealing both typical
# performance and variability, which a single split cannot provide.
print("Answer: B")


## Question 13: Cross-Validation Purpose

Your learning curve for logistic regression (Question 5) showed both training and validation curves converging quickly with a very small gap. Your manager asks: "Why didn't you just use a single train/test split to evaluate the model?" Which response is most accurate?

**A)** A single split is faster and produces the same ranking of models regardless of how the data is divided, so cross-validation is unnecessary computational overhead for most practical problems.

**B)** Cross-validation produces multiple performance estimates, revealing both typical performance and variability, which a single split cannot provide.

**C)** Cross-validation is only necessary for small datasets with fewer than 1,000 observations; larger datasets like this one do not benefit from it.

**D)** Cross-validation is primarily used to increase the training set size, which improves model accuracy compared to a single train/test split.

**E)** Cross-validation replaces the need for a final test set, so you can use all data for training and validation without holding anything out.


In [ ]:
# Question 14: Learning Curve Interpretation
# Answer: C
# Switch to a more flexible model family like tree-based methods, because the
# convergence at a moderate level indicates the linear model cannot capture the signal.
print("Answer: C")


## Question 14: Learning Curve Interpretation

Your learning curve (Question 5) showed logistic regression with both curves converging at approximately 0.86 ROC AUC with a gap of only 0.001. Based on this diagnosis, which action would most likely improve performance?

**A)** Collect more training data, because the validation curve has not yet plateaued and more data will continue improving generalization.

**B)** Increase regularization strength (lower C), because the small gap indicates overfitting that needs to be controlled.

**C)** Switch to a more flexible model family like tree-based methods, because the convergence at a moderate level indicates the linear model cannot capture the signal.

**D)** Reduce the number of features, because too many features are causing the model to underfit by spreading attention across irrelevant variables.

**E)** Increase max_iter for logistic regression, because the model has not converged during training and needs more optimization iterations.


In [ ]:
# Question 15: Validation Curve Interpretation
# Answer: B
# The validation curve shows that beyond depth 10, the model is overfitting --
# higher training scores reflect memorization of training data, not better generalization.
print("Answer: B")


## Question 15: Validation Curve Interpretation

Your validation curve for the decision tree (Question 6) showed training ROC AUC increasing steadily toward 1.0 while validation ROC AUC peaked around depth 10 and then plateaued. A colleague suggests using max_depth=20 because "the training score is higher." Why is this reasoning flawed?

**A)** Deeper trees take longer to train, so the computational cost outweighs any performance benefit at depth 20 compared to depth 10.

**B)** The validation curve shows that beyond depth 10, the model is overfitting -- higher training scores reflect memorization of training data, not better generalization.

**C)** Decision trees cannot effectively use depths greater than 15 because the tree structure becomes too fragmented to make reliable predictions.

**D)** The training score at depth 20 is artificially inflated because cross-validation leaks information from the validation folds into the training folds during the preprocessing and fitting steps, producing an optimistically biased estimate.

**E)** Deeper trees require more features than are available in this dataset, so depths beyond 10 produce degenerate splits.


In [ ]:
# Question 16: GridSearchCV vs RandomizedSearchCV
# Answer: B
# RandomizedSearchCV is preferred when the search space is large, because it provides
# a fixed computational budget regardless of the number of hyperparameters.
print("Answer: B")


## Question 16: GridSearchCV vs RandomizedSearchCV

In Questions 7 and 8, you used GridSearchCV for logistic regression and RandomizedSearchCV for gradient boosting. Your GridSearchCV tested 8 combinations while RandomizedSearchCV tested 20 out of 48 possible combinations. When should you prefer RandomizedSearchCV over GridSearchCV?

**A)** RandomizedSearchCV always finds better hyperparameters because random sampling explores the space more thoroughly than exhaustive search.

**B)** RandomizedSearchCV is preferred when the search space is large, because it provides a fixed computational budget regardless of the number of hyperparameters.

**C)** RandomizedSearchCV should only be used for ensemble methods that have many hyperparameters, while GridSearchCV is more appropriate for simpler linear models like logistic regression that have fewer parameters to tune.

**D)** RandomizedSearchCV is faster because it uses fewer cross-validation folds than GridSearchCV, reducing the total number of model fits.

**E)** RandomizedSearchCV is preferred when you need exact reproducibility, because grid search results vary depending on the order of evaluation.


In [ ]:
# Question 17: Fair Model Comparison
# Answer: C
# Fair comparison requires same evaluation conditions, and models should be tuned
# before comparison, since default settings favor some algorithms more than others.
print("Answer: C")


## Question 17: Fair Model Comparison

In your comparison table (Question 10), Random Forest had the highest cross-validated ROC AUC (0.9509) while Gradient Boosting had 0.9139 with default hyperparameters. However, after tuning (Question 8), Gradient Boosting achieved 0.9457. What is the most important lesson about fair model comparison?

**A)** Always choose the model with the highest ROC AUC, since ROC AUC is the most reliable metric for all classification problems regardless of business context.

**B)** Default hyperparameter comparisons provide a definitive ranking -- if a model performs poorly with defaults, tuning is unlikely to change the outcome significantly.

**C)** Fair comparison requires same evaluation conditions, and ideally each model should be tuned before final comparison, since default settings favor some algorithms more than others.

**D)** The model with the lowest standard deviation across folds is always the best choice because stability is more important than absolute performance.

**E)** Cross-validation comparison should always be followed by a statistical significance test; without it, any observed performance difference is unreliable.


In [ ]:
# Question 18: K-Fold Cross-Validation Mechanics
# Answer: A
# 5 training runs, each using 800 observations for training and 200 for validation.
print("Answer: A")


In [ ]:
# Question 19: Diagnosing Model Problems from Learning Curves
# Answer: B
# High variance; the model is memorizing training data, so reducing complexity
# or adding regularization would help.
print("Answer: B")


In [ ]:
# Question 20: Test Set Protocol
# Answer: C
# Repeatedly evaluating on the test set causes the selection process to overfit
# to the test data, making the reported score optimistically biased.
print("Answer: C")


In [ ]:
# Question 21: Choosing the Right Evaluation Metric
# Answer: C
# Recall (sensitivity), because it measures what fraction of actual positive cases
# the model identifies -- which is 0% here.
print("Answer: C")


In [ ]:
# Question 22: Data Leakage in Preprocessing
# Answer: C
# Fitting the scaler on all data before splitting leaked test set statistics into
# the preprocessing step, producing an inflated test score.
print("Answer: C")


## Closed-Resource Conceptual Questions (Q18–Q22)

The following five questions assess your conceptual understanding of the model evaluation, selection, and tuning principles covered in Chapter 15. These questions are designed for **closed-resource, in-class assessment** — answer them based on what you have learned from the chapter, without using AI tools, the textbook, or other resources.

Select the **best answer** for each question (A, B, C, D, or E).

---

## Question 18: K-Fold Cross-Validation Mechanics

In 5-fold cross-validation on a dataset of 1,000 observations, the data is divided into 5 equal parts. How many total model training runs occur, and how many observations are used for training in each run?

**A)** 5 training runs, each using 800 observations for training and 200 for validation.

**B)** 5 training runs, each using 200 observations for training and 800 for validation.

**C)** 10 training runs, each using 500 observations for training and 500 for validation.

**D)** 1 training run using all 1,000 observations, validated by splitting predictions into 5 groups.

**E)** 5 training runs, each using all 1,000 observations for both training and validation simultaneously.


## Question 19: Diagnosing Model Problems from Learning Curves

A model's learning curve shows that training accuracy is 0.98 while validation accuracy is 0.71 even when using the full training set. The gap between the two curves is not closing as more data is added. What does this indicate, and what is the most appropriate remedy?

**A)** High bias; the model is too simple and needs more features or a more flexible algorithm.

**B)** High variance; the model is memorizing training data, so reducing complexity or adding regularization would help.

**C)** Good fit; a training-validation gap is normal and expected for all well-performing models.

**D)** Insufficient data; collecting more training observations will eventually close the gap between the curves.

**E)** A preprocessing error; the large gap between training and validation indicates that the two sets have systematically different feature distributions.


## Question 20: Test Set Protocol

A data scientist evaluates 15 different model configurations on a held-out test set, selects the one with the best test performance, and reports that test score as the expected performance on new data. What is the fundamental problem with this approach?

**A)** Nothing is wrong; the test set provides an unbiased estimate for whichever model performs best on it.

**B)** Using 15 configurations is too few to find the best model; at least 50 should be tested on the held-out set.

**C)** Repeatedly evaluating on the test set causes the selection process to overfit to the test data, making the reported score optimistically biased.

**D)** The test set should always be larger than the training set to ensure statistically reliable evaluation when comparing multiple model configurations simultaneously.

**E)** The problem is computational inefficiency; evaluating on the test set is slower than using cross-validation on the training set.


## Question 21: Choosing the Right Evaluation Metric

A hospital builds a model to screen patients for a rare but life-threatening condition that affects 2% of patients. The model predicts "no risk" for every patient and achieves 98% accuracy. Which metric would best reveal this model's failure?

**A)** Accuracy, because it already shows the model is performing well at 98%.

**B)** Precision, because it measures what fraction of predicted positive cases are truly positive.

**C)** Recall (sensitivity), because it measures what fraction of actual positive cases the model identifies -- which is 0% here.

**D)** ROC AUC, because it evaluates the model's ranking quality across all possible classification thresholds.

**E)** Log loss, because it penalizes the model for assigning low probabilities to the true class.


## Question 22: Data Leakage in Preprocessing

A student fits a StandardScaler on the entire dataset before splitting into training and test sets. The model achieves ROC AUC of 0.95 on the test set but only 0.88 when deployed on truly new data. What caused this discrepancy?

**A)** The model needed more training iterations to converge properly before deployment.

**B)** StandardScaler is inappropriate for this type of dataset; a different scaler like MinMaxScaler or RobustScaler would produce consistent results and eliminate the discrepancy.

**C)** Fitting the scaler on all data before splitting leaked test set statistics into the preprocessing step, producing an inflated test score.

**D)** The deployment data must come from a different time period, which always causes performance drops unrelated to methodology.

**E)** The test set was too small, causing a random upward fluctuation that happened to reach 0.95 by chance.


# Summary and Reflection